In [1]:
# pipeline for this simsearch model
# what particularly simsearch does is that, 
# it's a self supervised learning model that learns about images from
# unlabelled dataset using embedding vector space and 
# when it came up with new images, it tries to match similarity
# of the new images with old images
# search similarity in embedding space

# project pipeline
# img -> self-supervised training -> encoder network -> img embeddings -> vector db -> similarity search

# SimSearch model training pipeline
# img -> augmentation -> encoder -> embeddings -> contrastive loss


In [2]:
import torch
print(torch.__version__)

2.10.0+cpu


In [3]:
# device configuration for model training
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cpu


In [4]:
# importing all necessary items for model training
from torch.utils.data import DataLoader

from models.model import SimSearchModel
from data.dataset import SimSearchDataset
from utils.augmentations import SimSearch_transform
from losses.nt_xnt import nt_xnt_loss


# creating dataset for model training
dataset = SimSearchDataset(
    root_dir="mammals",
    transform=SimSearch_transform
)

# dataloader for loading the data in model training
loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True
)

# SimSearchDataset returns view1, view2 (2 augmentation of same image)
# view1, view2 goes into DataLoader for loading into model


In [5]:
simsearchmodel = SimSearchModel().to(device=device)  # loading model into device

# optimizer for model training
optimizer=torch.optim.AdamW(
    simsearchmodel.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)


In [7]:
# SimSearch model training
def SimSearch_training(model, loader, loss_fn, optimizer_fn, device, epochs):
    
    for epoch in range(epochs):
        for view1, view2 in loader: # iterating through loader
            view1=view1.to(device) # loading view1 into device, similarly for view2
            view2=view2.to(device)

            z1=model(view1) # embedding for view1
            z2=model(view2) # embedding for view2

            loss=loss_fn(z1,z2) # calculating loss
            optimizer_fn.zero_grad() # zero gradient of optimizer
            loss.backward() # backpropagation
            optimizer_fn.step() # usign optimizer function
        print(f"Epoch: {epoch}| Loss: {loss.item()}")


SimSearch_training(model=simsearchmodel,loader=loader,loss_fn=nt_xnt_loss,optimizer_fn=optimizer,device=device,epochs=20)

# training needs a good compute power
# need something for it


Epoch: 0| Loss: 2.369154214859009
Epoch: 1| Loss: 2.483001947402954
Epoch: 2| Loss: 2.4166486263275146
Epoch: 3| Loss: 2.439300775527954
Epoch: 4| Loss: 2.4012749195098877
Epoch: 5| Loss: 2.259347915649414
Epoch: 6| Loss: 2.2148683071136475
Epoch: 7| Loss: 2.192564010620117
Epoch: 8| Loss: 2.236182928085327
Epoch: 9| Loss: 2.2942044734954834
Epoch: 10| Loss: 2.1208465099334717
Epoch: 11| Loss: 2.154994249343872
Epoch: 12| Loss: 2.1449172496795654
Epoch: 13| Loss: 2.1369447708129883
Epoch: 14| Loss: 2.071136713027954
Epoch: 15| Loss: 2.159576177597046
Epoch: 16| Loss: 2.1114144325256348
Epoch: 17| Loss: 2.2574684619903564
Epoch: 18| Loss: 2.0479979515075684
Epoch: 19| Loss: 2.0769612789154053


In [ ]:
torch.save(simsearchmodel.encoder.state_dict(),"SimSearch_0_encoder.pth") # saving the encoder of the model
#encoder generates embeddings of the images
